# 01. 컨텍스트와 토큰 예산

모델 호출에는 입력과 출력이 함께 들어갈 공간이 필요합니다. 이 노트북에서는 입력 길이를 호출 전에 추정하고, 실제 API 사용량과 비교합니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
# %pip은 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
# -q는 설치 로그를 줄이고, == 뒤 숫자는 수업에서 검증한 버전을 고정합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

아래 셀은 현재 작업 폴더에서 상위 폴더로 올라가며 `.env`를 찾습니다. 키 값은 화면에 출력하지 않습니다.

In [2]:
# find_dotenv는 현재 폴더부터 상위 폴더를 검색해 .env 경로를 찾습니다.
# model은 이후 셀에서 공통으로 사용할 LangChain 채팅 모델 객체입니다.
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
import os

# usecwd=True는 노트북을 실행한 폴더를 검색 시작점으로 지정합니다.
# 키 값은 출력하지 않고 존재 여부만 확인합니다.
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
# openai:는 공급자 이름이며, reasoning_effort="none"은 별도 추론 토큰을 사용하지 않는 설정입니다.
# max_tokens=300은 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none", max_tokens=300)
print("LangChain 모델 준비 완료")

LangChain 모델 준비 완료


## 2. 출력 공간을 먼저 예약합니다

전체 예산에서 출력 예약분과 안전 여유를 빼면 입력에 사용할 수 있는 범위가 나옵니다.

In [3]:
# total은 전체 한도, output_reserve는 답변용 예약분, safety_margin은 오차 여유입니다.
# 함수는 세 값을 계산한 입력 가용량을 반환하며, 음수이면 호출 전에 오류를 냅니다.
def input_budget(total: int, output_reserve: int, safety_margin: int) -> int:
    available = total - output_reserve - safety_margin
    if available < 0:
        raise ValueError("출력 예약분과 여유가 전체 예산보다 큽니다.")
    return available

available = input_budget(4096, 300, 128)
print("입력에 사용할 수 있는 토큰:", available)

입력에 사용할 수 있는 토큰: 3668


## 3. 로컬 추정값을 확인합니다

토크나이저의 추정값은 호출 전 차단에 쓰고, `AIMessage.usage_metadata`는 호출 후 기록에 씁니다.

In [4]:
# question과 evidence를 합쳐 모델에 보낼 prompt를 만듭니다.
# estimated는 prompt 문자열만 로컬에서 센 토큰 수입니다.
# system 메시지와 채팅 형식에 필요한 토큰은 제외되어 실제 API 입력 토큰과 다를 수 있습니다.
import tiktoken

question = "미르시립도서관의 정기 휴관일은 언제인가요?"
evidence = "[E01 p.2] 정기 휴관일은 매주 월요일입니다."
prompt = f"""근거:
{evidence}

질문: {question}"""
encoding = tiktoken.get_encoding("o200k_base")
estimated = len(encoding.encode(prompt))
print("로컬 추정 입력 토큰:", estimated)
print("예산 안에 있음:", estimated <= available)

로컬 추정 입력 토큰: 42
예산 안에 있음: True


## 4. 실제 API 사용량을 비교합니다

`AIMessage.text`와 `usage_metadata`를 함께 확인합니다. 출력은 실행마다 달라질 수 있습니다.

In [5]:
# messages는 역할과 내용을 짝지은 목록이며 system 지시를 user 질문보다 먼저 둡니다.
# invoke()는 모델을 한 번 호출하고, 반환된 AIMessage에서 text와 사용량을 읽습니다.
messages = [
    ("system", "근거에 있는 정보만 사용하고 출처를 표시하세요."),
    ("user", prompt),
]
response = model.invoke(messages)
# usage_metadata는 입력·출력 토큰 수를 담지만 제공되지 않을 가능성도 검사합니다.
usage = response.usage_metadata
if usage is None:
    raise RuntimeError("모델 사용량 메타데이터가 없습니다.")
print(response.text)
print("input_tokens:", usage.get("input_tokens"))
print("output_tokens:", usage.get("output_tokens"))

미르시립도서관의 정기 휴관일은 **매주 월요일**입니다. [E01 p.2]
input_tokens: 66
output_tokens: 32


## 정리

출력 공간을 먼저 예약하고, 입력을 로컬에서 추정한 뒤, 실제 사용량은 API 응답에서 확인합니다.

## 실패 재현: 출력 공간을 남기지 않습니다

전체 예산보다 출력 예약분과 여유를 크게 설정하면 입력에 사용할 공간이 음수가 됩니다.

In [6]:
# bad_total은 출력 예약분만큼만 잡아 안전 여유를 남기지 못한 잘못된 예산입니다.
# try/except는 예상한 ValueError만 포착하여 실패 원인을 출력합니다.
bad_total = 300
try:
    input_budget(bad_total, 300, 128)
except ValueError as error:
    print("관찰한 실패:", error)
else:
    raise AssertionError("예산 오류가 발생해야 합니다.")

관찰한 실패: 출력 예약분과 여유가 전체 예산보다 큽니다.


## 수정: 전체 예산을 복원합니다

출력 예약분과 여유를 제외하고도 입력 공간이 남도록 전체 예산을 4,096토큰으로 복원합니다.

In [7]:
# fixed_total을 충분히 늘려 출력 예약분과 안전 여유를 모두 확보합니다.
# fixed_available은 수정된 입력 가용량이며 assert로 계산 결과를 확인합니다.
fixed_total = 4096
fixed_available = input_budget(fixed_total, 300, 128)
assert fixed_available == 3668
print("수정 후 입력 예산:", fixed_available)

수정 후 입력 예산: 3668
